# Clase 200 — Kubernetes para servir modelos

Notebook **declarativo**: genera los 5 manifests YAML mínimos para desplegar el `iris-api` en K8s. Para verlos en vivo:

```bash
kind create cluster --name ml
kind load docker-image iris-api:v1 --name ml
kubectl apply -f k8s/
kubectl port-forward svc/iris-api 8000:80
```

## 🧠 Intuición previa

Antes de entrar a los YAML, quedate con la imagen mental de qué **es** Kubernetes:

> Kubernetes es un **orquestador** que mantiene *N* réplicas de tu servicio vivas y balancea la carga entre ellas — como el **director de una orquesta** que, si un músico falla a mitad de la obra, hace entrar al suplente sin que el público lo note.

De ahí salen todos los objetos que verás:

- **Deployment** = la partitura: "quiero 3 violinistas tocando esto". Vos declarás el *estado deseado* (`replicas: 3`), no los pasos para llegar.
- **Pod** = cada músico (una instancia de tu contenedor). Son **desechables**: si uno se cae, el director trae otro idéntico.
- **liveness/readiness probes** = el director mirando a cada músico: ¿sigue tocando? ¿está listo para recibir público? Si no, lo saca y reemplaza (self-healing).
- **Service** = el atril compartido: una IP estable que reparte las peticiones entre los músicos vivos, aunque cambien.
- **HorizontalPodAutoscaler** = contratar más músicos cuando el teatro se llena (sube la CPU/RPS) y despedirlos cuando se vacía.

La idea central: vos describís el **estado deseado** y Kubernetes trabaja sin parar para que la realidad lo iguale (*reconciliation loop*). No le decís "arrancá este proceso"; le decís "quiero que SIEMPRE haya 3 sanos".

In [ ]:
import os, shutil, tempfile
from pathlib import Path
WORK = Path(tempfile.gettempdir()) / 'k8s_demo'
if WORK.exists(): shutil.rmtree(WORK)
(WORK / 'k8s').mkdir(parents=True)
os.chdir(WORK)
print('cwd:', Path.cwd())

## 1. `Deployment` — 3 réplicas con probes y resources

In [ ]:
deploy = '''\
apiVersion: apps/v1
kind: Deployment
metadata:
  name: iris-api
  labels: { app: iris-api }
spec:
  replicas: 3
  strategy:
    type: RollingUpdate
    rollingUpdate: { maxSurge: 1, maxUnavailable: 0 }
  selector:
    matchLabels: { app: iris-api }
  template:
    metadata:
      labels: { app: iris-api }
    spec:
      containers:
        - name: api
          image: iris-api:v1
          imagePullPolicy: IfNotPresent
          ports:
            - containerPort: 8000
          resources:
            requests: { cpu: "100m", memory: "256Mi" }
            limits:   { cpu: "500m", memory: "512Mi" }
          startupProbe:
            httpGet: { path: /health, port: 8000 }
            failureThreshold: 30      # 30 * 2s = 60s para arrancar
            periodSeconds: 2
          livenessProbe:
            httpGet: { path: /health, port: 8000 }
            periodSeconds: 10
            failureThreshold: 3
          readinessProbe:
            httpGet: { path: /health, port: 8000 }
            periodSeconds: 5
            failureThreshold: 2
          securityContext:
            runAsNonRoot: true
            runAsUser: 1000
            allowPrivilegeEscalation: false
            readOnlyRootFilesystem: false
'''
Path('k8s/deployment.yaml').write_text(deploy)
print(deploy)

## 2. `Service` — load balancer interno

In [ ]:
svc = '''\
apiVersion: v1
kind: Service
metadata:
  name: iris-api
spec:
  type: ClusterIP
  selector: { app: iris-api }
  ports:
    - port: 80
      targetPort: 8000
      protocol: TCP
'''
Path('k8s/service.yaml').write_text(svc)
print(svc)

## 3. `HorizontalPodAutoscaler` — autoescalado CPU

In [ ]:
hpa = '''\
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: iris-api
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: iris-api
  minReplicas: 3
  maxReplicas: 10
  metrics:
    - type: Resource
      resource:
        name: cpu
        target: { type: Utilization, averageUtilization: 50 }
  behavior:
    scaleDown:
      stabilizationWindowSeconds: 300   # espera 5 min antes de bajar réplicas
    scaleUp:
      stabilizationWindowSeconds: 0     # sube ya
'''
Path('k8s/hpa.yaml').write_text(hpa)
print(hpa)

## 4. `Ingress` — exposición externa

In [ ]:
ing = '''\
apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: iris-api
  annotations:
    nginx.ingress.kubernetes.io/limit-rps: "100"
spec:
  ingressClassName: nginx
  rules:
    - host: iris.example.com
      http:
        paths:
          - path: /
            pathType: Prefix
            backend:
              service:
                name: iris-api
                port: { number: 80 }
'''
Path('k8s/ingress.yaml').write_text(ing)
print(ing)

## 5. `ConfigMap` + `Secret`

In [ ]:
cm = '''\
apiVersion: v1
kind: ConfigMap
metadata:
  name: iris-config
data:
  LOG_LEVEL: "info"
  MODEL_VERSION: "1.0.0"
---
apiVersion: v1
kind: Secret
metadata:
  name: iris-secrets
type: Opaque
stringData:
  API_KEY: "replace-me-via-external-secrets-operator"
'''
Path('k8s/config.yaml').write_text(cm)
print(cm)

## 6. Comandos kubectl de referencia

In [ ]:
commands = '''\
# Crear cluster local + cargar imagen
kind create cluster --name ml
kind load docker-image iris-api:v1 --name ml

# Apply
kubectl apply -f k8s/

# Estado
kubectl get deployments,pods,svc,hpa -l app=iris-api
kubectl describe pod -l app=iris-api
kubectl logs -l app=iris-api --tail=50

# Port-forward para probar local
kubectl port-forward svc/iris-api 8000:80
# en otra terminal: curl localhost:8000/predict -X POST -d ...

# Rolling update
kubectl set image deployment/iris-api api=iris-api:v2
kubectl rollout status deployment/iris-api
kubectl rollout history deployment/iris-api

# Rollback (instantáneo si v1 pods siguen referenciados)
kubectl rollout undo deployment/iris-api

# Force scale (puentea HPA momentáneamente)
kubectl scale deployment/iris-api --replicas=8

# Loadtest para disparar HPA
kubectl run loadtester --image=busybox -it --rm -- \\
    /bin/sh -c "while true; do wget -q -O- iris-api/health; done"
kubectl get hpa -w   # observá escalando

# Cleanup
kubectl delete -f k8s/
kind delete cluster --name ml
'''
print(commands)

## Ejercicio guiado

1. Levantá el cluster con `kind`, applies los 5 YAMLs, port-forward y curl al modelo.
2. Generá carga sintética y observá HPA escalar de 3 → 10 pods con `kubectl get hpa -w`.
3. Hacé un rolling update a una imagen rota (`iris-api:broken`). Confirmá que K8s detiene el rollout (no avanza si readiness falla). Rollback con `kubectl rollout undo`.
4. Cambiá `livenessProbe` a apuntar a `/wrong-path`. Observá `CrashLoopBackOff` en `kubectl get pods -w`. Revertí.
5. Bonus: agregá un `PodDisruptionBudget` con `minAvailable: 2` para garantizar disponibilidad durante drains de nodes.

## Conclusiones

- 5 manifests resuelven el 90% del caso ML serving: Deployment + Service + HPA + Ingress + (ConfigMap | Secret).
- Las 3 probes resuelven 3 problemas distintos — no son intercambiables.
- Sin `resources.requests`, los pods son ciudadanos de segunda; sin `limits`, son noisy neighbors.
- Rolling update + readiness probe = deploy sin downtime. Sin probe correcta = deploy con downtime invisible.

## ✅ Soluciones de los ejercicios

Soluciones de los 5 ejercicios del README. Kubernetes necesita un **cluster real** (`kind`/`kubectl`), así que los manifests y comandos se muestran como se ejecutarían, y **validamos los YAML con `pyyaml`** (el mismo parser del API server) y ejecutamos el *concepto* cuantificable: la **fórmula del HorizontalPodAutoscaler**, que es pura aritmética y no requiere cluster.

In [ ]:
import yaml, math
def valid_k8s(text):
    obj = yaml.safe_load(text)
    assert obj.get('apiVersion') and obj.get('kind'), 'todo objeto k8s tiene apiVersion + kind'
    return obj
print('validador de manifests listo.')

### Ejercicio 1 — Cluster local con `kind` (CLI de referencia)

Requiere Docker + kind. Comandos correctos; no ejecutables aquí.

In [ ]:
CLI = """
kind create cluster --name ml
kind load docker-image iris-api:v1 --name ml   # inyecta la imagen local al cluster
kubectl get nodes
"""
print(CLI)
print('kind crea un cluster k8s dentro de contenedores Docker (Kubernetes-IN-Docker).')

### Ejercicio 2 — Deployment + Service

El `Deployment` declara 3 réplicas con probes y `resources`; el `Service` les da una IP estable y balancea. Validamos ambos manifests.

In [ ]:
deployment = '''
apiVersion: apps/v1
kind: Deployment
metadata: {name: iris-api}
spec:
  replicas: 3
  selector: {matchLabels: {app: iris-api}}
  template:
    metadata: {labels: {app: iris-api}}
    spec:
      containers:
        - name: iris-api
          image: iris-api:v1
          ports: [{containerPort: 8000}]
          readinessProbe: {httpGet: {path: /health, port: 8000}, initialDelaySeconds: 3}
          livenessProbe:  {httpGet: {path: /health, port: 8000}, initialDelaySeconds: 10}
          resources:
            requests: {cpu: "250m", memory: "256Mi"}
            limits:   {cpu: "500m", memory: "512Mi"}
'''
service = '''
apiVersion: v1
kind: Service
metadata: {name: iris-api}
spec:
  selector: {app: iris-api}
  ports: [{port: 80, targetPort: 8000}]
'''
d = valid_k8s(deployment); s = valid_k8s(service)
assert d['spec']['replicas'] == 3
assert s['spec']['selector'] == d['spec']['template']['metadata']['labels'], 'el Service debe seleccionar los pods del Deployment'
print('CLI: kubectl apply -f deploy.yaml -f svc.yaml; kubectl get pods -w')
print('OK — Deployment(3 réplicas) + Service que las selecciona por label app=iris-api.')

### Ejercicio 3 — Probes y `CrashLoopBackOff`

Si la `livenessProbe` apunta a un endpoint inexistente, k8s cree que el pod está muerto, lo mata y reinicia en loop → `CrashLoopBackOff`. Mostramos el cambio y explicamos el mecanismo.

In [ ]:
broken = valid_k8s(deployment)
broken['spec']['template']['spec']['containers'][0]['livenessProbe']['httpGet']['path'] = '/wrong-endpoint'
# la probe fallará siempre -> restarts crecientes con backoff exponencial
assert broken['spec']['template']['spec']['containers'][0]['livenessProbe']['httpGet']['path'] == '/wrong-endpoint'
def backoff_schedule(n):     # k8s: 10s, 20s, 40s... hasta 5 min
    return [min(10 * 2 ** i, 300) for i in range(n)]
print('reinicios con backoff (s):', backoff_schedule(6))
print('kubectl get pods -w  =>  Running -> Error -> CrashLoopBackOff (revertir la probe lo cura).')

### Ejercicio 4 — HPA: la fórmula del autoescalado (ejecutable)

El HPA calcula `desiredReplicas = ceil(currentReplicas * currentMetric / targetMetric)`, acotado por `min`/`max`. Es aritmética pura: la implementamos y verificamos el escalado 3 → 10 bajo carga.

In [ ]:
hpa = '''
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata: {name: iris-api}
spec:
  scaleTargetRef: {apiVersion: apps/v1, kind: Deployment, name: iris-api}
  minReplicas: 3
  maxReplicas: 10
  metrics:
    - type: Resource
      resource: {name: cpu, target: {type: Utilization, averageUtilization: 50}}
'''
h = valid_k8s(hpa)
mn, mx = h['spec']['minReplicas'], h['spec']['maxReplicas']
target = h['spec']['metrics'][0]['resource']['target']['averageUtilization']

def desired_replicas(current, current_util, target=target, mn=mn, mx=mx):
    return max(mn, min(mx, math.ceil(current * current_util / target)))

# CPU al 50% target. Con carga al 100% de utilización actual y 3 réplicas -> quiere 6; sostenido escala al tope.
assert desired_replicas(3, 50) == 3, 'en target no escala'
assert desired_replicas(3, 100) == 6, '100/50 * 3 = 6'
assert desired_replicas(6, 100) == 10, 'topea en maxReplicas=10'
assert desired_replicas(10, 10) == 3, 'baja hasta minReplicas'
print('util 50% ->', desired_replicas(3, 50), '| util 100% ->', desired_replicas(3, 100),
      '| sostenido ->', desired_replicas(6, 100), '| ocioso ->', desired_replicas(10, 10))
print('OK — HPA escala 3->6->10 con carga y vuelve a 3 al vaciarse.')

### Ejercicio 5 — Rolling update + rollback

Un `RollingUpdate` reemplaza pods de a poco (`maxSurge`/`maxUnavailable`). Si la imagen `v2` está rota, las readiness probes nunca pasan, `rollout status` timeoutea, y `rollout undo` restaura `v1`. Simulamos la salud del rollout.

In [ ]:
CLI = """
kubectl set image deployment/iris-api iris-api=iris-api:v2
kubectl rollout status deployment/iris-api   # timeoutea si v2 no pasa readiness
kubectl rollout undo deployment/iris-api      # vuelve a v1
"""
def rollout_ok(new_pods_ready: bool):
    # k8s solo corta tráfico a los viejos cuando los nuevos pasan readiness
    return new_pods_ready

assert rollout_ok(new_pods_ready=False) is False, 'v2 rota => rollout NO progresa (protege el tráfico)'
assert rollout_ok(new_pods_ready=True) is True
print('v2 rota -> rollout status TIMEOUT -> kubectl rollout undo -> v1 restaurado (0 downtime).')
print('OK — el rolling update sano nunca deja el servicio sin pods listos.')